# EDA 06 — Pruning Anomaly Detection
The 2026 test set contains 9 fields with Near_Pruning_Flag=1. These post-pruning recovery fields have abnormally long harvest intervals and must be handled separately.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

BASE = Path(".")
df24 = pd.read_csv(BASE / "STEMS_Train_2024.csv")
df25 = pd.read_csv(BASE / "STEMS_Validate_2025.csv")
df26 = pd.read_csv(BASE / "STEMS_Test_2026.csv")

print("Near_Pruning_Flag distribution")
print(f"2024 Train    — Flag=0: {(df24['Near_Pruning_Flag']==0).sum()}  Flag=1: {(df24['Near_Pruning_Flag']==1).sum()}")
print(f"2025 Validate — Flag=0: {(df25['Near_Pruning_Flag']==0).sum()}  Flag=1: {(df25['Near_Pruning_Flag']==1).sum()}")
print(f"2026 Test     — Flag=0: {(df26['Near_Pruning_Flag']==0).sum()}  Flag=1: {(df26['Near_Pruning_Flag']==1).sum()}")


In [ ]:
flagged = df26[df26['Near_Pruning_Flag']==1][['Division','Field_No','Target_Days']]
clean   = df26[df26['Near_Pruning_Flag']==0][['Division','Field_No','Target_Days']]

print("=== FLAGGED FIELDS IN 2026 TEST ===")
print(flagged.to_string(index=False))
print(f"\nFlagged Target_Days range: {flagged['Target_Days'].min():.1f} — {flagged['Target_Days'].max():.1f} days")
print(f"Clean   Target_Days range: {clean['Target_Days'].min():.1f} — {clean['Target_Days'].max():.1f} days")


In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.hist(clean['Target_Days'],   bins=15, color='steelblue',  alpha=0.7, label='Normal (Flag=0)', edgecolor='k')
ax.hist(flagged['Target_Days'], bins=10, color='red',        alpha=0.7, label='Post-Pruning (Flag=1)', edgecolor='k')
ax.set_xlabel('Target_Days'); ax.set_ylabel('Count')
ax.set_title('2026 Test: Normal vs Post-Pruning Fields')
ax.legend(); ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('eda_06_pruning_anomalies.png', dpi=150)
plt.show()
print(f"\nConclusion: Flagged fields are biologically different — will be excluded from model evaluation.")
print(f"Test set: 71 rows → 62 clean rows after filtering.")


## Observations
- 2024 and 2025 have ZERO pruning-flagged fields — model has never seen this pattern
- 2026 has 9 flagged fields with Target_Days from 37.5 to 150 days (vs normal 10-30)
- These are post-pruning recovery fields — completely different biological process
- **Action: Filter Near_Pruning_Flag==1 from test set before evaluation**